# Fase 1: Entrenamiento, Calibración L-BFGS, Gráficas y Benchmark ReAct en BANKING77 (GPU T4)

Este notebook ejecuta de principio a fin la **Fase Experimental 1** sobre una GPU **NVIDIA T4** (Kaggle / Google Colab):

1. **Suite de Pruebas (`unittest`):** Verifica que los 19 tests unitarios y de integración pasen sin errores.
2. **Entrenamiento LoRA ($T = 1$):** Entrena el `DecisionModel` sobre **BANKING77** (77 clases) y exporta `best_lora_checkpoint/`, `val_logits.pt` y `test_logits.pt`.
3. **Calibración L-BFGS + Benchmark en Vivo ($T = 1$ vs. ReAct $T > 1$) + Gráficas:** Ajusta la temperatura óptima $T^*$, mide latencia *Time-to-Action* y fallos de JSON/alucinación en vivo, y genera las 3 figuras de publicación:
   - `reliability_diagram.png` (Diagramas de confiabilidad antes y después de L-BFGS)
   - `confidence_and_gating_curve.png` (Histograma de confianza y curva de la compuerta $\tau$)
   - `decision_vs_react_comparison.png` (Comparativa directa $T=1$ vs. ReAct $T>1$)
4. **Visualización Inline y Empaquetado ZIP:** Muestra las gráficas directamente en el notebook y comprime todo en `phase1_results.zip` listo para descargar.

## 1. Verificación del Hardware (GPU T4)

In [ ]:
!nvidia-smi

## 2. Clonar el Repositorio e Instalar Dependencias

> **Nota:** Si ya subiste la carpeta `PAM-F2` o clonaste el repositorio en Kaggle/Colab, asegúrate de estar dentro del directorio `PAM-F2`.

In [ ]:
# Si necesitas clonar desde GitHub, descomenta y ajusta tu URL:
# !git clone https://github.com/tu-usuario/AcecomF3.git
# %cd AcecomF3/PAM-F2

!pip install -q --upgrade pip
!pip install -q torch transformers peft accelerate datasets scipy scikit-learn matplotlib tqdm

## 3. Ejecutar la Suite de Tests Unitarios y de Integración

Verificamos que toda la arquitectura (`prompts`, `model`, `calibration`, `react_baseline`, `data`, `train`, `evaluate`) funcione correctamente antes de lanzar el entrenamiento.

In [ ]:
!PYTHONPATH=. python -m unittest discover -s tests -v

## 4. Entrenamiento del Modelo de Decisión ($T = 1$) con LoRA sobre BANKING77

Puedes elegir:
- `Qwen/Qwen2.5-0.5B-Instruct` (~8-12 min en T4).
- `Qwen/Qwen2.5-3B-Instruct` (~25-35 min en T4).

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"  # Cambia a "Qwen/Qwen2.5-3B-Instruct" si deseas el modelo de 3B
OUTPUT_DIR = "results/phase1_banking77"
EPOCHS = 3
BATCH_SIZE = 4
LR = 3e-4

!PYTHONPATH=. python -m src.phase1_banking77.train \
    --model $MODEL_NAME \
    --output-dir $OUTPUT_DIR \
    --epochs $EPOCHS \
    --batch-size $BATCH_SIZE \
    --lr $LR

## 5. Calibración Post-Hoc (L-BFGS), Benchmark en Vivo ($T=1$ vs. ReAct $T>1$) y Generación de Figuras

Este comando:
1. Ejecuta el benchmark en vivo sobre el mismo modelo y GPU comparando **Paso Único ($T=1$)** frente a **Decodificación Autorregresiva ReAct ($T>1$)** (`--run-live-benchmark`).
2. Optimiza la temperatura $T^*$ mediante **L-BFGS** sobre los logits de validación (`val_logits.pt`).
3. Evalúa **Accuracy, ECE, MCE y NLL** sobre el conjunto de test (`test_logits.pt`).
4. Exporta `phase1_metrics.json`, `benchmark_t1_vs_react.json` y las 3 figuras PNG.

In [ ]:
BENCHMARK_SAMPLES = 50

!PYTHONPATH=. python -m src.phase1_banking77.evaluate \
    --results-dir $OUTPUT_DIR \
    --model $MODEL_NAME \
    --run-live-benchmark \
    --benchmark-samples $BENCHMARK_SAMPLES

## 6. Visualizar Métricas y Gráficas Generadas en el Notebook

In [ ]:
import json
from pathlib import Path
from IPython.display import Image, display

res_dir = Path(OUTPUT_DIR)
metrics_path = res_dir / "phase1_metrics.json"

if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
    print("=== RESUMEN DE MÉTRICAS FASE 1 (BANKING77) ===")
    print(f"Temperatura Óptima L-BFGS (T*): {metrics['optimal_temperature']:.4f}")
    print(f"Test Accuracy:                 {metrics['test_accuracy'] * 100:.2f}%")
    print(f"Test ECE (Antes -> Después):   {metrics['test_ece_before']:.4f} -> {metrics['test_ece_after']:.4f}")
    print(f"Meta ECE < 0.05 Cumplida:      {metrics['ece_target_met']}")
    print(f"Latencia Media T=1:            {metrics['decision_model_latency_ms']:.2f} ms")
    if metrics.get("react_comparison"):
        rc = metrics["react_comparison"]
        print(f"Latencia Media ReAct (T>1):    {rc['mean_latency_ms']:.2f} ms ({rc['mean_generated_tokens']:.1f} tokens)")
        print(f"Fallos Sintaxis/Alucinación:   T=1: 0.00% vs ReAct: {rc['invalid_total_rate'] * 100:.2f}%")

for fig_name in [
    "reliability_diagram.png",
    "confidence_and_gating_curve.png",
    "decision_vs_react_comparison.png",
]:
    fig_file = res_dir / fig_name
    if fig_file.exists():
        print(f"\n--- {fig_name} ---")
        display(Image(filename=str(fig_file)))

## 7. Empaquetar Resultados en `phase1_results.zip` para Descarga

In [ ]:
!zip -r phase1_results.zip $OUTPUT_DIR
!ls -lh phase1_results.zip